In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 

from datetime import datetime 
import xgboost as xgb

import os

In [4]:
# data = ['transaction_id', 'is_fraud', 'created_at', 'is_subscription', 'transaction_type',
#         'currency_amount', 'currency_id', 'amount_scaled', 'merchant_customer_id',
#         'merchant_country', 'ip_address', 'platform', 'merchant_id', 'merchant_shop_id',
#         'merchant_shop_name', 'is_secured', 'ip_country', 'payment_type', 'card_id', 
#         'bank', 'cardcountry', 'bin', 'card_holder_first_name', 'card_holder_last_name']

data = ['transaction_id', 'is_fraud', 'created_at', 'is_subscription', 'transaction_type',
        'currency_amount', 'currency_id', 'merchant_customer_id',
        'merchant_customer_email', 'merchant_country', 'merchant_language', 'ip_address',
        'platform', 'merchant_id', 'merchant_shop_id', 'merchant_shop_name', 'is_secured',
        'ip_country', 'payment_type', 'user_agent', 'card_id', 'bank', 'cardbrand',
        'cardcountry', 'cardtype', 'bin', 'card_exp_relative', 'card_holder_first_name',
        'card_holder_last_name']

In [ ]:
!python --version

In [5]:
train_path = '/kaggle/input/int20h_test_2025/train.csv' 

In [6]:
total_rows = sum(1 for _ in open(train_path)) - 1  # -1 for header
indices_to_skip = set(np.random.choice(
    range(1, total_rows + 1),  # Skip from row 1 (after header)
    size=int(total_rows * 0.58),  # Skip 58% of rows - 19.3 gb
    replace=False
))

# Read CSV skipping the selected rows
train = pd.read_csv(train_path, 
                 skiprows=lambda x: x in indices_to_skip, usecols=data)

In [6]:
train.head()

,transaction_id,is_fraud,created_at,is_subscription,transaction_type,currency_amount,currency_id,merchant_customer_id,merchant_customer_email,merchant_country,...,user_agent,card_id,bank,cardbrand,cardcountry,cardtype,bin,card_exp_relative,card_holder_first_name,card_holder_last_name
8880,15372181325488238308,0,2024-01-24 06:34:28.253682787,False,token,5815.8,122,19f25045f8b88283a5f62d8b53248d78b528c16dbcfec7...,db30f9b09fc6db70bd69492bad0993610e034dd0aaf196...,USA,...,AQA user agent,9e2d16884ca5bce237e5eb9541d2bc0a6a7203052f8126...,CITIZENS STATE BANK,VISA,USA,DEBIT,03b78aebb24bbaabeddcb2909a7f25ad498ace0d260562...,62.0,beee81f790a2244d244e464aa0b43fb938369932ccdb1f...,671e5bac76139268d3b6c3a669d69a54f809d5529f1560...
14264,17090834222135584838,0,2024-01-28 06:34:09.253682787,False,first,5815.8,122,1307987255064842c9d85377b38ad0f2f01b450fa7de53...,db30f9b09fc6db70bd69492bad0993610e034dd0aaf196...,USA,...,AQA user agent,4d0c82a87774f03102b5feaee9144740615bffcba0bb98...,FINSTELLA LTD,MASTERCARD,LUX,CREDIT,64707889ecc86bfe9d5e23a2cd899e99c8e7cdf1d899fc...,69.0,4fac22cc347af210cc96e6aaf122d5484ccfc9b7044d38...,a73e67029e39b7eb05e89d79ac6cbba81308ecabbb7143...
14610,2355393699365161926,0,2024-01-24 12:45:19.253682787,False,first,5815.8,122,e7f3ca1e8a5cea91436cb55cf4142e423448f37cf7f000...,db30f9b09fc6db70bd69492bad0993610e034dd0aaf196...,USA,...,AQA user agent,9e2d16884ca5bce237e5eb9541d2bc0a6a7203052f8126...,CITIZENS STATE BANK,VISA,USA,DEBIT,03b78aebb24bbaabeddcb2909a7f25ad498ace0d260562...,62.0,beee81f790a2244d244e464aa0b43fb938369932ccdb1f...,671e5bac76139268d3b6c3a669d69a54f809d5529f1560...
14633,5445521635572745384,0,2024-01-24 13:32:20.253682787,False,first,135.0,122,30465eb63dd9e1f432ea3867c1d3503122a2743b11a116...,a4ddc3a7d3776333dd31e2700dfc23b8e63ea6586fdfe0...,BRA,...,C,f44f6030f90e5df882dd95d7638aca78ccdae0c96137ea...,CONOTOXIA SP. Z O.O,VISA,POL,DEBIT,8fe9bba2fca44c2ab8feba35b66030fe6276941d46a115...,23.0,8a454891ff1c61aca27cf97303bbd908af8dad7a488361...,ef258560343665e70405b99fa4a295f8e701ea046d1502...
14879,4569912325797039117,0,2024-01-22 05:25:45.253682787,False,first,13500.0,122,ef258560343665e70405b99fa4a295f8e701ea046d1502...,cc4683e6e68e4a540b66a148e2c87ab2a85e3413a9e3e0...,GBR,...,a,17317b7d2585c5c3fb80e21519c35cf88dab5c79251c49...,ITAU UNIBANCO S.A.,MASTERCARD,BRA,CREDIT,1bf337d81c854c09c405ae264b76c71470da3a7461ac91...,9.0,8a454891ff1c61aca27cf97303bbd908af8dad7a488361...,ef258560343665e70405b99fa4a295f8e701ea046d1502...


In [ ]:
train.isnull().sum()

In [7]:
train['is_fraud'].value_counts(normalize=True)

is_fraud
0    0.972312
1    0.027688
Name: proportion, dtype: float64

In [7]:
train.dropna(inplace=True)

In [ ]:
train['is_fraud'].value_counts(normalize=True)

In [ ]:
train.describe()

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt

def analyze_correlations_in_chunks(file_path, chunk_size=5000000):
    first_chunk = True
    correlation_sum = None
    n_chunks = 0
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        numeric_chunk = chunk.select_dtypes(include=[np.number])
        chunk_corr = numeric_chunk.corr()
        
        if first_chunk:
            correlation_sum = chunk_corr
            first_chunk = False
        else:
            correlation_sum += chunk_corr
        
        n_chunks += 1
    
    return correlation_sum / n_chunks

def plot_correlation_matrix(correlation_matrix):
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
def analyze_fraud_patterns(file_path, target_column='is_fraud', chunk_size=5000000):
    # Initialize statistics dictionaries
    feature_stats = {}
    class_distributions = {0: 0, 1: 0}
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        # Update class distribution
        class_counts = chunk[target_column].value_counts()
        for label in class_counts.index:
            class_distributions[label] += class_counts[label]
        
        # Calculate statistics for numeric columns
        numeric_chunk = chunk.select_dtypes(include=[np.number])
        for column in numeric_chunk.columns:
            if column != target_column:  # Skip the target column itself
                if column not in feature_stats:
                    feature_stats[column] = {
                        'mean_fraud': [],
                        'mean_normal': [],
                        'std_fraud': [],
                        'std_normal': []
                    }
                
                # Calculate statistics by class
                fraud_stats = numeric_chunk[chunk[target_column] == 1][column].agg(['mean', 'std'])
                normal_stats = numeric_chunk[chunk[target_column] == 0][column].agg(['mean', 'std'])
                
                feature_stats[column]['mean_fraud'].append(fraud_stats['mean'])
                feature_stats[column]['mean_normal'].append(normal_stats['mean'])
                feature_stats[column]['std_fraud'].append(fraud_stats['std'])
                feature_stats[column]['std_normal'].append(normal_stats['std'])
    
    return feature_stats, class_distributions

In [ ]:
def analyze_outliers_by_class(file_path, target_column='is_fraud', chunk_size=5000000):
    outlier_stats = {}
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        numeric_chunk = chunk.select_dtypes(include=[np.number])
        
        for column in numeric_chunk.columns:
            if column != target_column:
                if column not in outlier_stats:
                    outlier_stats[column] = {'fraud_outliers': 0, 'normal_outliers': 0, 'total_rows': 0}
                
                # Calculate IQR for outlier detection
                Q1 = numeric_chunk[column].quantile(0.25)
                Q3 = numeric_chunk[column].quantile(0.75)
                IQR = Q3 - Q1
                
                # Count outliers by class
                outliers = (numeric_chunk[column] < (Q1 - 1.5 * IQR)) | (numeric_chunk[column] > (Q3 + 1.5 * IQR))
                outlier_stats[column]['fraud_outliers'] += sum(outliers & (chunk[target_column] == 1))
                outlier_stats[column]['normal_outliers'] += sum(outliers & (chunk[target_column] == 0))
                outlier_stats[column]['total_rows'] += len(chunk)
    
    return outlier_stats

In [ ]:
# Run the analyses


# 1. Correlation Analysis
correlation_matrix = analyze_correlations_in_chunks(train_path)
plot_correlation_matrix(correlation_matrix)

In [ ]:
# 2. Fraud Patterns Analysis
feature_stats, class_distribution = analyze_fraud_patterns(train_path)

In [ ]:
# 3. Outlier Analysis
outlier_stats = analyze_outliers_by_class(train_path)

In [ ]:
print("\nOutlier Analysis:")
for column, stats in outlier_stats.items():
    fraud_rate = stats['fraud_outliers'] / stats['total_rows'] * 100
    normal_rate = stats['normal_outliers'] / stats['total_rows'] * 100
    print(f"{column}: Fraud outlier rate: {fraud_rate:.2f}%, Normal outlier rate: {normal_rate:.2f}%")

In [ ]:
# Print key findings
print("\nClass Distribution (Fraud vs Normal):")
print(class_distribution)

In [ ]:
print("\nFeature Statistics:")
for feature, stats in feature_stats.items():
    print(f"\n{feature}:")
    print(f"Fraud transactions - Mean: {np.mean(stats['mean_fraud']):.3f}, Std: {np.mean(stats['std_fraud']):.3f}")
    print(f"Normal transactions - Mean: {np.mean(stats['mean_normal']):.3f}, Std: {np.mean(stats['std_normal']):.3f}")

In [8]:
def create_amount_features(df):
    # Transaction amount percentiles/stats per merchant
    df['amount_to_merchant_mean'] = df.groupby('merchant_id')['currency_amount'].transform('mean')
    df['amount_to_merchant_std'] = df.groupby('merchant_id')['currency_amount'].transform('std')
    
    # Ratio features
    df['amount_to_merchant_ratio'] = df['currency_amount'] / (df['amount_to_merchant_mean'] + 1e-8)

In [9]:
def create_merchant_features(df):
    # Merchant transaction frequency
    df['merchant_tx_count'] = df.groupby('merchant_id')['transaction_id'].transform('count')
    
    # Merchant-currency patterns
    df['merchant_currency_ratio'] = df.groupby(['merchant_id', 'currency_id'])['transaction_id'].transform('count') / df['merchant_tx_count']
    
    # Merchant shop patterns
    df['merchant_shop_ratio'] = df.groupby(['merchant_id', 'merchant_shop_id'])['transaction_id'].transform('count') / df['merchant_tx_count']

In [10]:
def create_card_features(df):
    # Card usage patterns
    df['card_merchant_count'] = df.groupby('card_exp_relative')['merchant_id'].transform('nunique')
    df['card_currency_count'] = df.groupby('card_exp_relative')['currency_id'].transform('nunique')
    
    # Card amount patterns
    df['card_amount_mean'] = df.groupby('card_exp_relative')['currency_amount'].transform('mean')
    df['card_amount_std'] = df.groupby('card_exp_relative')['currency_amount'].transform('std')

In [11]:
def encode_categorical_features(df):
    from sklearn.preprocessing import LabelEncoder
    
    # Target encoding for high-cardinality features
    df['browser_fraud_rate'] = df.groupby('browser')['is_fraud'].transform('mean')
    df['os_fraud_rate'] = df.groupby('operating_system')['is_fraud'].transform('mean')
    
    # Frequency encoding
    df['browser_freq'] = df.groupby('browser')['transaction_id'].transform('count') / len(df)
    df['os_freq'] = df.groupby('operating_system')['transaction_id'].transform('count') / len(df)

In [12]:
def create_interaction_features(df):
    # Interaction between amount and merchant features
    df['amount_merchant_interaction'] = df['currency_amount'] * df['merchant_tx_count']
    
    # Currency-amount interactions
    df['currency_amount_interaction'] = df['currency_id'].astype(str) + '_' + df['currency_amount'].astype(str)
    
    # Time-based interactions (if you have timestamp)
    # df['hour_amount_interaction'] = df['hour'] * df['amount_scaled']

In [13]:
def create_anomaly_features(df):
    from scipy import stats
    
    # Z-score for amounts
    df['amount_zscore'] = stats.zscore(df['currency_amount'])
    
    # Isolation Forest anomaly score
    from sklearn.ensemble import IsolationForest
    df['isolation_score'] = IsolationForest().fit_predict(df[['currency_amount', 'merchant_tx_count']])

In [14]:
def engineer_features(df):
    df = df.copy()
    
    create_amount_features(df)
    create_merchant_features(df)
    create_card_features(df)
    # encode_categorical_features(df)
    create_interaction_features(df)
    create_anomaly_features(df)
    
    return df

# Apply feature engineering
df_engineered = engineer_features(train)

In [15]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

def train_xgboost(train_path, chunk_size=5000000):
    # Create DMatrix in chunks
    dtrain = None
    print("Creating DMatrix...")
    
    for chunk in pd.read_csv(train_path, chunksize=chunk_size):
        # Apply feature engineering
        chunk = engineer_features(chunk)
        
        # Prepare features
        X_chunk = chunk.select_dtypes(include=[np.number])
        y_chunk = X_chunk.pop('is_fraud')
        
        # Convert to DMatrix
        if dtrain is None:
            dtrain = xgb.DMatrix(X_chunk, label=y_chunk)
        else:
            dtrain.extend(xgb.DMatrix(X_chunk, label=y_chunk))
    
    # Set parameters for fraud detection
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'scale_pos_weight': class_ratio,  # handle class imbalance
        'max_depth': 6,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'tree_method': 'hist',  # for faster training
        'grow_policy': 'lossguide',  # for better performance
        'max_bin': 256  # for memory efficiency
    }
    
    # Train model
    print("Training XGBoost...")
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=[(dtrain, 'train')],
        early_stopping_rounds=10
    )
    
    return model

In [16]:
test_path = '/kaggle/input/int20h_test_2025/test.csv'

In [ ]:
def evaluate_xgboost(model, test_path, chunk_size=5000000):
    y_true_all = []
    y_pred_all = []
    
    print("Evaluating model...")
    for chunk in pd.read_csv(test_path, chunksize=chunk_size):
        # Apply feature engineering
        chunk = engineer_features(chunk)
        
        # Prepare features
        X_chunk = chunk.select_dtypes(include=[np.number])
        y_chunk = X_chunk.pop('is_fraud')
        
        # Predict
        dtest = xgb.DMatrix(X_chunk)
        y_pred = model.predict(dtest)
        
        # Collect results
        y_true_all.extend(y_chunk)
        y_pred_all.extend(y_pred > 0.5)  # Convert probabilities to binary predictions
    
    # Print evaluation metrics
    print("\nClassification Report:")
    print(classification_report(y_true_all, y_pred_all))
    
    print("\nROC AUC Score:")
    print(roc_auc_score(y_true_all, y_pred_all))

In [ ]:
# Feature importance analysis
def analyze_feature_importance(model, feature_names):
    importance = model.get_score(importance_type='gain')
    importance = pd.DataFrame.from_dict(importance, orient='index', columns=['importance'])
    importance = importance.sort_values('importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(importance.head(10))

куку

In [29]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score
import gc

def train_memory_efficient_xgboost(train_path, test_path, chunk_size=10000):
    print("Starting memory-efficient training...")
    
    # Read first chunk and get numeric features only
    first_chunk = next(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False))
    
    # Explicitly select numeric columns and remove categorical ones
    numeric_features = first_chunk.select_dtypes(include=[np.number]).columns.tolist()
    # Remove specific categorical columns and target variable
    exclude_columns = ['payment_type', 'transaction_source', 'browser', 
                      'browser_version', 'operating_system', 
                      'operating_system_version', 'is_fraud']
    feature_names = [col for col in numeric_features if col not in exclude_columns]
    
    print(f"Using numeric features only: {feature_names}")
    
    del first_chunk
    gc.collect()
    
    # Initialize model
    model = xgb.XGBClassifier(
        tree_method='hist',
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100,
        use_label_encoder=False,
        n_jobs=-1
    )
    
    print("Training on chunks...")
    for i, chunk in enumerate(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False)):
        if i >= 50:
            break
            
        X_chunk = chunk[feature_names]
        y_chunk = chunk['is_fraud']
        
        model.fit(X_chunk, y_chunk, xgb_model=model.get_booster() if i > 0 else None)
        
        del X_chunk, y_chunk, chunk
        gc.collect()
        
        if i % 10 == 0:
            print(f"Processed {i} chunks...")
    
    print("Making predictions on test data...")
    predictions = []
    
    for i, test_chunk in enumerate(pd.read_csv(test_path, chunksize=chunk_size, low_memory=False)):
        if i >= 10:
            break
            
        X_test = test_chunk[feature_names]
        y_pred = model.predict(X_test)
        
        # Create predictions DataFrame
        pred_df = pd.DataFrame({
            'prediction': y_pred
        })
        predictions.append(pred_df)
        
        del X_test, test_chunk
        gc.collect()
        
        if i % 10 == 0:
            print(f"Processed {i} test chunks...")
    
    # Combine all predictions
    final_predictions = pd.concat(predictions, axis=0, ignore_index=True)
    print(f"\nTotal predictions made: {len(final_predictions)}")
    
    # Print feature importance
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    })
    print("\nFeature Importance:")
    print(importance_df.sort_values('importance', ascending=False))
    
    return model, final_predictions



In [30]:
# Run training
try:
    print("Starting training process...")
    model, predictions = train_memory_efficient_xgboost(train_path, test_path)
    
    # Save model
    model.save_model('fraud_detection_model.json')
    print("\nModel saved to fraud_detection_model.json")
    
    # Save predictions
    predictions.to_csv('test_predictions.csv', index=False)
    print("Predictions saved to test_predictions.csv")
    
    # Print prediction statistics
    print("\nPrediction Statistics:")
    print(f"Number of fraudulent transactions predicted: {sum(predictions['prediction'] == 1)}")
    print(f"Number of normal transactions predicted: {sum(predictions['prediction'] == 0)}")
    
except Exception as e:
    print(f"An error occurred: {str(e)}")
    import traceback
    print(traceback.format_exc())

Starting training process...
Starting memory-efficient training...
Using numeric features only: ['transaction_id', 'currency_amount', 'currency_id', 'amount_scaled', 'merchant_id', 'merchant_shop_id', 'order_number', 'card_exp_relative']
Training on chunks...
Processed 0 chunks...
Processed 10 chunks...
Processed 20 chunks...
Processed 30 chunks...
Processed 40 chunks...
Making predictions on test data...
Processed 0 test chunks...

Total predictions made: 100000

Feature Importance:
             feature  importance
2        currency_id    0.255373
1    currency_amount    0.130114
3      amount_scaled    0.120532
5   merchant_shop_id    0.119857
4        merchant_id    0.107686
7  card_exp_relative    0.094597
0     transaction_id    0.092509
6       order_number    0.079332

Model saved to fraud_detection_model.json
Predictions saved to test_predictions.csv

Prediction Statistics:
Number of fraudulent transactions predicted: 1650
Number of normal transactions predicted: 98350


In [31]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import f1_score, classification_report
import gc

def train_memory_efficient_xgboost(train_path, test_path, chunk_size=10000):
   print("Starting memory-efficient training...")
   
   # Read first chunk and get numeric features only
   first_chunk = next(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False))
   
   # Explicitly select numeric columns and remove categorical ones
   numeric_features = first_chunk.select_dtypes(include=[np.number]).columns.tolist()
   exclude_columns = ['payment_type', 'transaction_source', 'browser', 
                     'browser_version', 'operating_system', 
                     'operating_system_version', 'is_fraud']
   feature_names = [col for col in numeric_features if col not in exclude_columns]
   
   print(f"Using numeric features only: {feature_names}")
   
   del first_chunk
   gc.collect()
   
   # Initialize model
   model = xgb.XGBClassifier(
       tree_method='hist',
       max_depth=6,
       learning_rate=0.1,
       n_estimators=100,
       use_label_encoder=False,
       n_jobs=-1
   )
   
   print("Training and evaluating on validation set...")
   validation_predictions = []
   validation_true = []
   
   # Use last 5 chunks for validation
   for i, chunk in enumerate(pd.read_csv(train_path, chunksize=chunk_size, low_memory=False)):
       if i < 45:  # First 45 chunks for training
           X_chunk = chunk[feature_names]
           y_chunk = chunk['is_fraud']
           model.fit(X_chunk, y_chunk, xgb_model=model.get_booster() if i > 0 else None)
           
           if i % 10 == 0:
               print(f"Processed {i} training chunks...")
               
       elif i < 50:  # Last 5 chunks for validation
           X_val = chunk[feature_names]
           y_val = chunk['is_fraud']
           y_pred = model.predict(X_val)
           
           validation_predictions.extend(y_pred)
           validation_true.extend(y_val)
           
           print(f"Processed validation chunk {i}")
           
       else:
           break
           
       # Free memory
       del chunk
       gc.collect()
   
   # Calculate and print validation metrics
   print("\nValidation Metrics:")
   print(classification_report(validation_true, validation_predictions))
   macro_f1 = f1_score(validation_true, validation_predictions, average='macro')
   print(f"Validation Macro F1 Score: {macro_f1:.4f}")
   
   print("\nMaking predictions on test data...")
   predictions = []
   
   for i, test_chunk in enumerate(pd.read_csv(test_path, chunksize=chunk_size, low_memory=False)):
       if i >= 10:
           break
           
       X_test = test_chunk[feature_names]
       y_pred = model.predict(X_test)
       
       # Create predictions DataFrame
       pred_df = pd.DataFrame({
           'prediction': y_pred
       })
       predictions.append(pred_df)
       
       del X_test, test_chunk
       gc.collect()
       
       if i % 10 == 0:
           print(f"Processed {i} test chunks...")
   
   # Combine all predictions
   final_predictions = pd.concat(predictions, axis=0, ignore_index=True)
   print(f"\nTotal predictions made: {len(final_predictions)}")
   
   # Print feature importance
   importance_df = pd.DataFrame({
       'feature': feature_names,
       'importance': model.feature_importances_
   })
   print("\nFeature Importance:")
   print(importance_df.sort_values('importance', ascending=False))
   
   return model, final_predictions, macro_f1



In [32]:
# Run training
try:
   print("Starting training process...")
   model, predictions, macro_f1 = train_memory_efficient_xgboost(train_path, test_path)
   
   # Save model
   model.save_model('fraud_detection_model.json')
   print("\nModel saved to fraud_detection_model.json")
   
   # Save predictions
   predictions.to_csv('test_predictions.csv', index=False)
   print("Predictions saved to test_predictions.csv")
   
   # Print prediction statistics
   print("\nPrediction Statistics:")
   print(f"Number of fraudulent transactions predicted: {sum(predictions['prediction'] == 1)}")
   print(f"Number of normal transactions predicted: {sum(predictions['prediction'] == 0)}")
   print(f"Final Validation Macro F1 Score: {macro_f1:.4f}")
   
except Exception as e:
   print(f"An error occurred: {str(e)}")
   import traceback
   print(traceback.format_exc())

Starting training process...
Starting memory-efficient training...
Using numeric features only: ['transaction_id', 'currency_amount', 'currency_id', 'amount_scaled', 'merchant_id', 'merchant_shop_id', 'order_number', 'card_exp_relative']
Training and evaluating on validation set...
Processed 0 training chunks...
Processed 10 training chunks...
Processed 20 training chunks...
Processed 30 training chunks...
Processed 40 training chunks...
Processed validation chunk 45
Processed validation chunk 46
Processed validation chunk 47
Processed validation chunk 48
Processed validation chunk 49

Validation Metrics:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96     47567
           1       0.11      0.10      0.10      2433

    accuracy                           0.92     50000
   macro avg       0.53      0.53      0.53     50000
weighted avg       0.91      0.92      0.91     50000

Validation Macro F1 Score: 0.5285

Making predictions on tes